# Adaptive Rolling Dynamic RFM — H&M dataset (Google Colab)

Notebook này chạy bản nghiên cứu Adaptive Rolling Dynamic RFM trên bộ H&M Personalized Fashion Recommendations.

**Pipeline:** giao dịch CSV → DuckDB tổng hợp `customer-day` → RFM theo snapshot → cửa sổ cố định 4/8/13 tuần và cửa sổ thích ứng → MiniBatch K-Means → metric và phân tích chuyển cụm.

## Trước khi chạy
- Chọn **Runtime → Change runtime type → CPU**; không cần GPU.
- Chạy từng cell từ trên xuống.
- Dataset giao dịch khoảng 3,2 GiB và khách hàng khoảng 200 MB sẽ được tải trực tiếp từ Hugging Face vào bộ nhớ tạm `/content` của Colab. Cell tải sẽ kiểm tra dung lượng từ API và có thể tiếp tục tải file chưa hoàn tất.
- Colab `/content` là lưu trữ tạm. Cell cuối cùng cho phép sao chép kết quả sang Google Drive.
- Chạy quick test trước. Dù chỉ tính 2 snapshot, lần đầu vẫn cần đọc toàn bộ transaction CSV để tạo Parquet trung gian.

**Lưu ý phương pháp:** H&M không có `basket_id/order_id`. Trong code, F được định nghĩa là số ngày mua hàng phân biệt trong cửa sổ; không được mô tả thành số đơn hàng chính xác. Silhouette và Davies–Bouldin là metric phân cụm, không phải accuracy.


In [ ]:
%pip -q install "duckdb>=1.1.3" "pandas>=2.2.0" "numpy>=1.26.0" "scikit-learn>=1.5.0" "pyarrow>=17.0.0" "joblib>=1.4.0"

import sys, duckdb, pandas as pd, numpy as np, sklearn, pyarrow
print("Python:", sys.version.split()[0])
print("DuckDB:", duckdb.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)


In [ ]:
from pathlib import Path
import json

PROJECT_DIR = Path("/content/adaptive_rolling_dynamic_rfm")
SRC_DIR = PROJECT_DIR / "src"
SRC_DIR.mkdir(parents=True, exist_ok=True)
SCRIPT_PATH = SRC_DIR / "adaptive_dynamic_rfm.py"
SCRIPT_CONTENT = "#!/usr/bin/env python\n\"\"\"Adaptive Rolling Dynamic RFM for the H&M Personalized Fashion Recommendations data.\n\nThis is an independent research implementation inspired by public H&M customer\nsegmentation / RFM pipelines. It does not modify or copy files from the reference\nrepository. Designed for Windows PowerShell and the original H&M CSV schema.\n\nMain design:\n- Aggregate transaction lines to customer-day first using DuckDB (large-file friendly).\n- At each chronological snapshot, estimate a customer-specific window from the\n  median inter-purchase-day gap observed in the trailing 365 days.\n- Clip adaptive window to [28, 91] days; use 56 days with insufficient history.\n- Compute R plus fixed 28/56/91-day F/M and adaptive-window F/M.\n- H&M has no basket/order ID. Main F is therefore distinct purchase-days (a proxy);\n  item-line count is also retained as a sensitivity feature.\n- Fit a reference MiniBatchKMeans model on the first snapshot, then predict later\n  snapshots with fixed centroids so cluster labels are comparable over time.\n- Report sampled Silhouette and Davies-Bouldin metrics and cluster transition counts.\n\nNo model performance is claimed until the script is run on the user's data.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport logging\nimport math\nimport sys\nfrom datetime import date, timedelta\nfrom pathlib import Path\nfrom typing import Dict, List, Tuple\n\nimport duckdb\nimport numpy as np\nimport pandas as pd\nfrom sklearn.cluster import MiniBatchKMeans\nfrom sklearn.metrics import davies_bouldin_score, silhouette_score\nfrom sklearn.preprocessing import StandardScaler\nimport joblib\n\nLOG = logging.getLogger(\"adaptive_dynamic_rfm\")\n\nFIXED_WINDOWS = {\n    \"fixed_4w\": (\"f_28d\", \"m_28d\", \"lines_28d\"),\n    \"fixed_8w\": (\"f_56d\", \"m_56d\", \"lines_56d\"),\n    \"fixed_13w\": (\"f_91d\", \"m_91d\", \"lines_91d\"),\n    \"adaptive\": (\"f_adaptive\", \"m_adaptive\", \"lines_adaptive\"),\n}\n\n\ndef sql_quote(value: str | Path) -> str:\n    \"\"\"Return a safely single-quoted SQL string literal for a local path.\"\"\"\n    return \"'\" + str(value).replace(\"\\\\\", \"/\").replace(\"'\", \"''\") + \"'\"\n\n\ndef make_connection(db_path: Path, memory_limit: str, threads: int) -> duckdb.DuckDBPyConnection:\n    db_path.parent.mkdir(parents=True, exist_ok=True)\n    con = duckdb.connect(str(db_path))\n    con.execute(f\"SET memory_limit = {sql_quote(memory_limit)}\")\n    con.execute(f\"SET threads = {int(threads)}\")\n    con.execute(\"SET preserve_insertion_order = false\")\n    return con\n\n\ndef build_daily_parquet(\n    con: duckdb.DuckDBPyConnection,\n    transactions_csv: Path,\n    daily_parquet: Path,\n    rebuild: bool = False,\n) -> None:\n    \"\"\"Read the large CSV once and aggregate transaction lines to customer-day.\"\"\"\n    if daily_parquet.exists() and not rebuild:\n        LOG.info(\"Reusing existing daily aggregate: %s\", daily_parquet)\n        return\n    if not transactions_csv.exists():\n        raise FileNotFoundError(f\"Transactions CSV not found: {transactions_csv}\")\n\n    daily_parquet.parent.mkdir(parents=True, exist_ok=True)\n    if daily_parquet.exists():\n        daily_parquet.unlink()\n\n    csv_sql = sql_quote(transactions_csv)\n    out_sql = sql_quote(daily_parquet)\n    LOG.info(\"Aggregating the H&M CSV to customer-day Parquet; this reads the large CSV once.\")\n    query = f\"\"\"\n        COPY (\n            SELECT\n                TRY_CAST(t_dat AS DATE) AS t_dat,\n                CAST(customer_id AS VARCHAR) AS customer_id,\n                SUM(TRY_CAST(price AS DOUBLE))::DOUBLE AS spend,\n                COUNT(*)::BIGINT AS purchase_lines,\n                COUNT(DISTINCT article_id)::BIGINT AS unique_articles\n            FROM read_csv_auto({csv_sql}, header = true, all_varchar = true,\n                               sample_size = 100000, ignore_errors = false)\n            WHERE customer_id IS NOT NULL\n              AND CAST(customer_id AS VARCHAR) <> ''\n              AND TRY_CAST(t_dat AS DATE) IS NOT NULL\n              AND TRY_CAST(price AS DOUBLE) IS NOT NULL\n              AND TRY_CAST(price AS DOUBLE) >= 0\n            GROUP BY 1, 2\n        ) TO {out_sql} (FORMAT PARQUET, COMPRESSION ZSTD)\n    \"\"\"\n    con.execute(query)\n    stats = con.execute(\n        f\"SELECT COUNT(*) AS customer_days, COUNT(DISTINCT customer_id) AS customers, \"\n        f\"MIN(t_dat) AS min_date, MAX(t_dat) AS max_date, SUM(purchase_lines) AS input_lines \"\n        f\"FROM read_parquet({out_sql})\"\n    ).fetchone()\n    LOG.info(\n        \"Daily aggregate ready: %s customer-days, %s customers, date range %s to %s, %s input rows.\",\n        f\"{stats[0]:,}\", f\"{stats[1]:,}\", stats[2], stats[3], f\"{stats[4]:,}\",\n    )\n\n\ndef query_date_range(con: duckdb.DuckDBPyConnection, daily_parquet: Path) -> Tuple[date, date, int]:\n    row = con.execute(\n        f\"SELECT MIN(t_dat), MAX(t_dat), COUNT(DISTINCT customer_id) \"\n        f\"FROM read_parquet({sql_quote(daily_parquet)})\"\n    ).fetchone()\n    if not row or row[0] is None:\n        raise ValueError(\"No valid transactions found after parsing the CSV.\")\n    return row[0], row[1], int(row[2])\n\n\ndef generate_snapshots(start: date, end: date, warmup_days: int, every_days: int) -> List[date]:\n    first = start + timedelta(days=warmup_days)\n    if first > end:\n        raise ValueError(\n            f\"Not enough history: data ends {end}, but warm-up requires a snapshot after {first}. \"\n            \"Try --warmup-days 90 for a shorter initial test.\"\n        )\n    snapshots: List[date] = []\n    current = first\n    while current <= end:\n        snapshots.append(current)\n        current += timedelta(days=every_days)\n    if snapshots[-1] != end:\n        snapshots.append(end)\n    return snapshots\n\n\ndef build_snapshot_query(daily_parquet: Path, cutoff: date, gap_lookback_days: int,\n                         min_window_days: int, max_window_days: int,\n                         default_window_days: int, gap_multiplier: float) -> str:\n    \"\"\"Build one leakage-safe snapshot query; all adaptive-window history ends at cutoff.\"\"\"\n    daily = sql_quote(daily_parquet)\n    cut = cutoff.isoformat()\n    return f\"\"\"\n        WITH gap_history AS (\n            SELECT customer_id, t_dat,\n                   LAG(t_dat) OVER (PARTITION BY customer_id ORDER BY t_dat) AS previous_date\n            FROM read_parquet({daily})\n            WHERE t_dat <= DATE '{cut}'\n              AND t_dat > DATE '{cut}' - INTERVAL '{int(gap_lookback_days)} days'\n        ),\n        purchase_gaps AS (\n            SELECT customer_id,\n                   DATE_DIFF('day', previous_date, t_dat)::DOUBLE AS gap_days\n            FROM gap_history\n            WHERE previous_date IS NOT NULL\n              AND DATE_DIFF('day', previous_date, t_dat) > 0\n        ),\n        customer_gap_stats AS (\n            SELECT customer_id, COUNT(*) AS n_gaps, MEDIAN(gap_days) AS median_gap_days\n            FROM purchase_gaps\n            GROUP BY customer_id\n        ),\n        adaptive_windows AS (\n            SELECT customer_id,\n                CASE\n                    WHEN n_gaps >= 2 THEN\n                        GREATEST({int(min_window_days)}, LEAST({int(max_window_days)},\n                            CAST(CEIL(median_gap_days * {float(gap_multiplier)}) AS INTEGER)))\n                    ELSE {int(default_window_days)}\n                END AS adaptive_window_days\n            FROM customer_gap_stats\n        ),\n        last_purchase AS (\n            SELECT customer_id, MAX(t_dat) AS last_purchase_date\n            FROM read_parquet({daily})\n            WHERE t_dat <= DATE '{cut}'\n            GROUP BY customer_id\n        ),\n        customer_base AS (\n            SELECT l.customer_id,\n                   DATE_DIFF('day', l.last_purchase_date, DATE '{cut}')::BIGINT AS recency_days,\n                   COALESCE(w.adaptive_window_days, {int(default_window_days)})::INTEGER AS adaptive_window_days\n            FROM last_purchase l\n            LEFT JOIN adaptive_windows w USING (customer_id)\n        ),\n        recent_daily AS (\n            SELECT d.customer_id, d.t_dat, d.spend, d.purchase_lines,\n                   b.recency_days, b.adaptive_window_days\n            FROM customer_base b\n            LEFT JOIN read_parquet({daily}) d\n              ON d.customer_id = b.customer_id\n             AND d.t_dat <= DATE '{cut}'\n             AND d.t_dat > DATE '{cut}' - INTERVAL '{int(max_window_days)} days'\n        )\n        SELECT customer_id,\n               MAX(recency_days)::BIGINT AS recency_days,\n               MAX(adaptive_window_days)::INTEGER AS adaptive_window_days,\n\n               COUNT(DISTINCT CASE WHEN t_dat > DATE '{cut}' - INTERVAL '28 days'\n                                   THEN t_dat END)::BIGINT AS f_28d,\n               SUM(CASE WHEN t_dat > DATE '{cut}' - INTERVAL '28 days'\n                        THEN spend ELSE 0 END)::DOUBLE AS m_28d,\n               SUM(CASE WHEN t_dat > DATE '{cut}' - INTERVAL '28 days'\n                        THEN purchase_lines ELSE 0 END)::BIGINT AS lines_28d,\n\n               COUNT(DISTINCT CASE WHEN t_dat > DATE '{cut}' - INTERVAL '56 days'\n                                   THEN t_dat END)::BIGINT AS f_56d,\n               SUM(CASE WHEN t_dat > DATE '{cut}' - INTERVAL '56 days'\n                        THEN spend ELSE 0 END)::DOUBLE AS m_56d,\n               SUM(CASE WHEN t_dat > DATE '{cut}' - INTERVAL '56 days'\n                        THEN purchase_lines ELSE 0 END)::BIGINT AS lines_56d,\n\n               COUNT(DISTINCT CASE WHEN t_dat > DATE '{cut}' - INTERVAL '91 days'\n                                   THEN t_dat END)::BIGINT AS f_91d,\n               SUM(CASE WHEN t_dat > DATE '{cut}' - INTERVAL '91 days'\n                        THEN spend ELSE 0 END)::DOUBLE AS m_91d,\n               SUM(CASE WHEN t_dat > DATE '{cut}' - INTERVAL '91 days'\n                        THEN purchase_lines ELSE 0 END)::BIGINT AS lines_91d,\n\n               COUNT(DISTINCT CASE\n                    WHEN t_dat IS NOT NULL\n                     AND DATE_DIFF('day', t_dat, DATE '{cut}') < adaptive_window_days\n                    THEN t_dat END)::BIGINT AS f_adaptive,\n               SUM(CASE WHEN t_dat IS NOT NULL\n                         AND DATE_DIFF('day', t_dat, DATE '{cut}') < adaptive_window_days\n                        THEN spend ELSE 0 END)::DOUBLE AS m_adaptive,\n               SUM(CASE WHEN t_dat IS NOT NULL\n                         AND DATE_DIFF('day', t_dat, DATE '{cut}') < adaptive_window_days\n                        THEN purchase_lines ELSE 0 END)::BIGINT AS lines_adaptive\n        FROM recent_daily\n        GROUP BY customer_id\n    \"\"\"\n\n\ndef make_features(df: pd.DataFrame, f_col: str, m_col: str) -> np.ndarray:\n    \"\"\"Log-transform skewed R/F/M features and return float32 for memory efficiency.\"\"\"\n    recency = np.log1p(np.maximum(df[\"recency_days\"].to_numpy(dtype=np.float32), 0))\n    freq = np.log1p(np.maximum(df[f_col].to_numpy(dtype=np.float32), 0))\n    monetary = np.log1p(np.maximum(df[m_col].to_numpy(dtype=np.float32), 0))\n    return np.column_stack((recency, freq, monetary)).astype(np.float32, copy=False)\n\n\ndef score_sample(X: np.ndarray, labels: np.ndarray, sample_size: int, seed: int) -> Tuple[float, float]:\n    \"\"\"Compute internal clustering metrics on a fixed-size sample to limit runtime.\"\"\"\n    n = X.shape[0]\n    if n < 3 or np.unique(labels).size < 2:\n        return float(\"nan\"), float(\"nan\")\n    rng = np.random.default_rng(seed)\n    idx = rng.choice(n, size=min(sample_size, n), replace=False)\n    xs, ys = X[idx], labels[idx]\n    if np.unique(ys).size < 2:\n        return float(\"nan\"), float(\"nan\")\n    try:\n        sil = float(silhouette_score(xs, ys, metric=\"euclidean\"))\n    except Exception:\n        sil = float(\"nan\")\n    try:\n        dbi = float(davies_bouldin_score(xs, ys))\n    except Exception:\n        dbi = float(\"nan\")\n    return sil, dbi\n\n\ndef run_experiment(args: argparse.Namespace) -> None:\n    out_dir = Path(args.output_dir).expanduser().resolve()\n    out_dir.mkdir(parents=True, exist_ok=True)\n    work_dir = out_dir / \"work\"\n    work_dir.mkdir(parents=True, exist_ok=True)\n    daily_parquet = work_dir / \"customer_day.parquet\"\n    db_file = work_dir / \"adaptive_rfm.duckdb\"\n\n    con = make_connection(db_file, args.memory_limit, args.threads)\n    try:\n        build_daily_parquet(con, Path(args.transactions).expanduser().resolve(), daily_parquet,\n                            rebuild=args.rebuild_daily)\n        min_date, max_date, total_customers = query_date_range(con, daily_parquet)\n        snapshots = generate_snapshots(min_date, max_date, args.warmup_days, args.snapshot_every_days)\n        if args.max_snapshots and args.max_snapshots > 0:\n            snapshots = snapshots[:args.max_snapshots]\n        LOG.info(\"Input dates: %s to %s; %s unique customers.\", min_date, max_date, f\"{total_customers:,}\")\n        LOG.info(\"Snapshots (%s): %s\", len(snapshots), \", \".join(d.isoformat() for d in snapshots))\n\n        model_names = list(FIXED_WINDOWS.keys())\n        scalers: Dict[str, StandardScaler] = {}\n        models: Dict[str, MiniBatchKMeans] = {}\n        metric_rows: List[dict] = []\n        snapshot_files: List[Tuple[date, Path]] = []\n\n        for snap_i, cutoff in enumerate(snapshots):\n            LOG.info(\"Snapshot %s/%s at %s\", snap_i + 1, len(snapshots), cutoff.isoformat())\n            sql = build_snapshot_query(\n                daily_parquet, cutoff, args.gap_lookback_days,\n                args.min_window_days, args.max_window_days,\n                args.default_window_days, args.gap_multiplier,\n            )\n            df = con.execute(sql).df()\n            if df.empty:\n                LOG.warning(\"No customers found at cutoff %s; skipping.\", cutoff)\n                continue\n            # Make sure aggregate outputs are clean and deterministic.\n            num_cols = [c for c in df.columns if c != \"customer_id\"]\n            df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0)\n            snapshot_file = out_dir / f\"snapshot_{cutoff.strftime('%Y%m%d')}.parquet\"\n\n            for model_name, (f_col, m_col, _lines_col) in FIXED_WINDOWS.items():\n                X_raw = make_features(df, f_col, m_col)\n                if model_name not in scalers:\n                    scaler = StandardScaler()\n                    X = scaler.fit_transform(X_raw).astype(np.float32, copy=False)\n                    if len(df) < args.clusters:\n                        raise ValueError(f\"Only {len(df)} customers at baseline; need at least {args.clusters} clusters.\")\n                    model = MiniBatchKMeans(\n                        n_clusters=args.clusters,\n                        random_state=args.random_state,\n                        batch_size=args.batch_size,\n                        n_init=3,\n                        max_iter=args.max_iter,\n                        reassignment_ratio=0.01,\n                    )\n                    LOG.info(\"Fitting reference clusters for %s on first snapshot.\", model_name)\n                    model.fit(X)\n                    scalers[model_name] = scaler\n                    models[model_name] = model\n                else:\n                    X = scalers[model_name].transform(X_raw).astype(np.float32, copy=False)\n                    model = models[model_name]\n                labels = model.predict(X).astype(np.int16, copy=False)\n                cluster_col = f\"cluster_{model_name}\"\n                df[cluster_col] = labels\n                sil, dbi = score_sample(X, labels, args.metric_sample_size, args.random_state + snap_i)\n                metric_rows.append({\n                    \"snapshot_date\": cutoff.isoformat(),\n                    \"model\": model_name,\n                    \"customers\": int(len(df)),\n                    \"clusters\": int(np.unique(labels).size),\n                    \"silhouette_sample\": sil,\n                    \"davies_bouldin_sample\": dbi,\n                    \"adaptive_window_median_days\": float(df[\"adaptive_window_days\"].median()),\n                    \"adaptive_window_min_days\": int(df[\"adaptive_window_days\"].min()),\n                    \"adaptive_window_max_days\": int(df[\"adaptive_window_days\"].max()),\n                    \"frequency_definition\": \"distinct_purchase_days\",\n                })\n                del X_raw, X, labels\n\n            df[\"snapshot_date\"] = cutoff.isoformat()\n            df.to_parquet(snapshot_file, index=False, compression=\"zstd\")\n            snapshot_files.append((cutoff, snapshot_file))\n            LOG.info(\"Wrote %s (%s customer rows)\", snapshot_file.name, f\"{len(df):,}\")\n            del df\n\n        if not snapshot_files:\n            raise RuntimeError(\"No snapshot outputs were produced.\")\n\n        metrics_path = out_dir / \"clustering_metrics.csv\"\n        pd.DataFrame(metric_rows).to_csv(metrics_path, index=False)\n        LOG.info(\"Wrote metrics: %s\", metrics_path)\n\n        # Transition counts are calculated for customers present in consecutive snapshots.\n        transition_rows: List[pd.DataFrame] = []\n        for (prev_date, prev_file), (curr_date, curr_file) in zip(snapshot_files, snapshot_files[1:]):\n            for model_name in model_names:\n                col = f\"cluster_{model_name}\"\n                query = f\"\"\"\n                    SELECT a.{col} AS from_cluster,\n                           b.{col} AS to_cluster,\n                           COUNT(*)::BIGINT AS customers\n                    FROM read_parquet({sql_quote(prev_file)}) a\n                    INNER JOIN read_parquet({sql_quote(curr_file)}) b USING (customer_id)\n                    GROUP BY 1, 2\n                    ORDER BY 1, 2\n                \"\"\"\n                tr = con.execute(query).df()\n                if not tr.empty:\n                    tr[\"model\"] = model_name\n                    tr[\"from_snapshot\"] = prev_date.isoformat()\n                    tr[\"to_snapshot\"] = curr_date.isoformat()\n                    transition_rows.append(tr)\n        transitions_path = out_dir / \"cluster_transitions.csv\"\n        if transition_rows:\n            pd.concat(transition_rows, ignore_index=True).to_csv(transitions_path, index=False)\n        else:\n            pd.DataFrame(columns=[\"from_cluster\", \"to_cluster\", \"customers\", \"model\", \"from_snapshot\", \"to_snapshot\"]).to_csv(transitions_path, index=False)\n        LOG.info(\"Wrote transitions: %s\", transitions_path)\n\n        # Customer-profile interpretation for the last snapshot, if customers.csv was provided.\n        if args.customers:\n            customer_csv = Path(args.customers).expanduser().resolve()\n            if customer_csv.exists():\n                last_file = snapshot_files[-1][1]\n                profile_path = out_dir / \"adaptive_cluster_profile.csv\"\n                q = f\"\"\"\n                    COPY (\n                        SELECT s.cluster_adaptive,\n                               COUNT(*)::BIGINT AS customer_count,\n                               MEDIAN(TRY_CAST(c.age AS DOUBLE)) AS median_age,\n                               AVG(TRY_CAST(c.age AS DOUBLE)) AS mean_age,\n                               AVG(TRY_CAST(c.Active AS DOUBLE)) AS active_rate,\n                               AVG(TRY_CAST(c.FN AS DOUBLE)) AS fashion_news_flag_rate\n                        FROM read_parquet({sql_quote(last_file)}) s\n                        LEFT JOIN read_csv_auto({sql_quote(customer_csv)}, header=true, all_varchar=true) c\n                          ON s.customer_id = c.customer_id\n                        GROUP BY s.cluster_adaptive\n                        ORDER BY s.cluster_adaptive\n                    ) TO {sql_quote(profile_path)} (HEADER, DELIMITER ',')\n                \"\"\"\n                con.execute(q)\n                LOG.info(\"Wrote latest adaptive-cluster profile: %s\", profile_path)\n\n        # Persist reference cluster models and metadata for reproducibility.\n        joblib.dump({\"scalers\": scalers, \"models\": models}, out_dir / \"reference_cluster_models.joblib\")\n        metadata = {\n            \"method\": \"Adaptive Rolling Dynamic RFM\",\n            \"dataset\": \"H&M Personalized Fashion Recommendations\",\n            \"transaction_source\": str(Path(args.transactions).expanduser().resolve()),\n            \"daily_aggregate\": str(daily_parquet),\n            \"min_date\": min_date.isoformat(),\n            \"max_date\": max_date.isoformat(),\n            \"snapshot_dates\": [d.isoformat() for d, _ in snapshot_files],\n            \"snapshot_every_days\": args.snapshot_every_days,\n            \"warmup_days\": args.warmup_days,\n            \"gap_lookback_days\": args.gap_lookback_days,\n            \"adaptive_window_rule\": f\"clip(ceil({args.gap_multiplier} * median inter-purchase-day gap), {args.min_window_days}, {args.max_window_days}); fallback={args.default_window_days} days when fewer than 2 gaps\",\n            \"R_definition\": \"days since last purchase as of snapshot cutoff, using all prior history\",\n            \"F_definition\": \"number of distinct purchase dates in the window; H&M has no basket/order ID\",\n            \"M_definition\": \"sum of per-item-line price in the window\",\n            \"extra_frequency_sensitivity_features\": \"lines_28d, lines_56d, lines_91d, lines_adaptive\",\n            \"clustering\": f\"MiniBatchKMeans(k={args.clusters}); reference centroids fitted on first snapshot and held fixed for temporal label comparability\",\n            \"metrics\": \"Silhouette and Davies-Bouldin on a deterministic sample; not classification accuracy\",\n            \"note\": \"No performance result is asserted until run on the complete dataset.\",\n        }\n        (out_dir / \"run_metadata.json\").write_text(json.dumps(metadata, indent=2), encoding=\"utf-8\")\n        LOG.info(\"Finished. Output directory: %s\", out_dir)\n    finally:\n        con.close()\n\n\ndef parse_args(argv: List[str] | None = None) -> argparse.Namespace:\n    p = argparse.ArgumentParser(description=\"Adaptive Rolling Dynamic RFM on H&M transactions.\")\n    p.add_argument(\"--transactions\", required=True, help=\"Path to transactions_train.csv\")\n    p.add_argument(\"--customers\", default=None, help=\"Optional path to customers.csv for profile summaries\")\n    p.add_argument(\"--output-dir\", required=True, help=\"Directory for Parquet snapshots and CSV reports\")\n    p.add_argument(\"--rebuild-daily\", action=\"store_true\", help=\"Rebuild customer_day.parquet from CSV\")\n    p.add_argument(\"--warmup-days\", type=int, default=180, help=\"History required before the first snapshot\")\n    p.add_argument(\"--snapshot-every-days\", type=int, default=28, help=\"Snapshot cadence; default approximately every 4 weeks\")\n    p.add_argument(\"--gap-lookback-days\", type=int, default=365, help=\"Past-only lookback for estimating purchase rhythm\")\n    p.add_argument(\"--min-window-days\", type=int, default=28, help=\"Minimum adaptive window length\")\n    p.add_argument(\"--max-window-days\", type=int, default=91, help=\"Maximum adaptive window length\")\n    p.add_argument(\"--default-window-days\", type=int, default=56, help=\"Fallback adaptive window for sparse histories\")\n    p.add_argument(\"--gap-multiplier\", type=float, default=2.0, help=\"Adaptive window = multiplier * median interpurchase gap\")\n    p.add_argument(\"--clusters\", type=int, default=5, help=\"Number of reference clusters\")\n    p.add_argument(\"--batch-size\", type=int, default=8192, help=\"MiniBatchKMeans batch size\")\n    p.add_argument(\"--max-iter\", type=int, default=100, help=\"MiniBatchKMeans maximum iterations\")\n    p.add_argument(\"--metric-sample-size\", type=int, default=10000, help=\"Number of customers sampled for internal clustering metrics\")\n    p.add_argument(\"--random-state\", type=int, default=42)\n    p.add_argument(\"--threads\", type=int, default=4, help=\"DuckDB thread count\")\n    p.add_argument(\"--memory-limit\", default=\"4GB\", help=\"DuckDB memory limit, e.g. 4GB or 8GB\")\n    p.add_argument(\"--max-snapshots\", type=int, default=0, help=\"Optional quick test: only run first N snapshots; 0 means all\")\n    p.add_argument(\"--verbose\", action=\"store_true\")\n    args = p.parse_args(argv)\n    if args.snapshot_every_days < 1:\n        p.error(\"--snapshot-every-days must be >= 1\")\n    if args.min_window_days < 1 or args.max_window_days < args.min_window_days:\n        p.error(\"Invalid adaptive window bounds\")\n    if args.default_window_days < args.min_window_days or args.default_window_days > args.max_window_days:\n        p.error(\"--default-window-days must be between min and max adaptive window\")\n    if args.gap_multiplier <= 0:\n        p.error(\"--gap-multiplier must be positive\")\n    return args\n\n\ndef main(argv: List[str] | None = None) -> int:\n    args = parse_args(argv)\n    logging.basicConfig(\n        level=logging.DEBUG if args.verbose else logging.INFO,\n        format=\"%(asctime)s | %(levelname)s | %(message)s\",\n        datefmt=\"%Y-%m-%d %H:%M:%S\",\n    )\n    try:\n        run_experiment(args)\n    except KeyboardInterrupt:\n        LOG.error(\"Interrupted. Customer-day Parquet can be reused on the next run.\")\n        return 130\n    except Exception as exc:\n        LOG.exception(\"Run failed: %s\", exc)\n        return 1\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"
SCRIPT_PATH.write_text(SCRIPT_CONTENT, encoding="utf-8")
print("Wrote research script:", SCRIPT_PATH)
print("Script bytes:", SCRIPT_PATH.stat().st_size)


In [ ]:
from pathlib import Path
import subprocess, requests, json, os

DATA_DIR = Path("/content/hm_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
HF_REPO = "einrafh/hnm-fashion-recommendations-data"
HF_API = f"https://huggingface.co/api/datasets/{HF_REPO}/tree/main?recursive=true&expand=false"
HF_BASE = f"https://huggingface.co/datasets/{HF_REPO}/resolve/main/data/raw"

# The current RFM pipeline uses transactions_train.csv and customers.csv.
# articles.csv is optional and is not needed by this implementation.
FILES = {
    "transactions_train.csv": f"{HF_BASE}/transactions_train.csv",
    "customers.csv": f"{HF_BASE}/customers.csv",
}

print("Reading public dataset file sizes from Hugging Face API...")
r = requests.get(HF_API, timeout=60)
r.raise_for_status()
items = r.json()
remote_sizes = {x.get("path", ""): int(x.get("size", 0)) for x in items if x.get("type") == "file"}

for name, url in FILES.items():
    dst = DATA_DIR / name
    remote_path = f"data/raw/{name}"
    expected = remote_sizes.get(remote_path, 0)
    if expected == 0:
        print(f"Warning: remote size not found for {name}; curl will still try to download.")
    if dst.exists() and expected and dst.stat().st_size == expected:
        print(f"Already complete: {name} ({dst.stat().st_size:,} bytes)")
        continue
    print(f"Downloading {name}; expected bytes: {expected:,}" if expected else f"Downloading {name}")
    # Resume from the current file size if Colab disconnected during a prior download.
    cmd = ["curl", "-fL", "--retry", "5", "-C", "-", "-o", str(dst), url]
    subprocess.run(cmd, check=True)
    actual = dst.stat().st_size if dst.exists() else 0
    print(f"Finished {name}: {actual:,} bytes")
    if expected and actual != expected:
        raise RuntimeError(f"Size mismatch for {name}: expected {expected:,}, got {actual:,}. Re-run this cell to resume.")

print("\nData files ready:")
for p in DATA_DIR.iterdir():
    print(p.name, f"{p.stat().st_size / (1024**3):.2f} GiB")


In [ ]:
# Inspect the header and a few rows only; do not load the 3+ GB transactions CSV into Pandas.
import pandas as pd

transactions_path = DATA_DIR / "transactions_train.csv"
customers_path = DATA_DIR / "customers.csv"

transactions_sample = pd.read_csv(transactions_path, nrows=5)
customers_sample = pd.read_csv(customers_path, nrows=5)
print("Transactions columns:", transactions_sample.columns.tolist())
print("Customers columns:", customers_sample.columns.tolist())
display(transactions_sample)
display(customers_sample)

required = {"t_dat", "customer_id", "article_id", "price"}
missing = required - set(transactions_sample.columns)
if missing:
    raise ValueError(f"Unexpected H&M schema; missing required columns: {sorted(missing)}")
print("Schema check passed.")


## Quick test: 2 snapshots

Lần đầu, DuckDB sẽ quét file giao dịch lớn để tạo `customer_day.parquet`; bước này là phần tốn thời gian chính. Sau đó code chỉ tính 2 snapshot để xác minh pipeline. Thư mục output được giữ trong `/content/hm_adaptive_rfm_output`, để lần chạy tiếp theo có thể tái sử dụng Parquet tổng hợp.


In [ ]:
import subprocess, sys
from pathlib import Path

OUTPUT_DIR = Path("/content/hm_adaptive_rfm_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

quick_cmd = [
    sys.executable, str(SCRIPT_PATH),
    "--transactions", str(DATA_DIR / "transactions_train.csv"),
    "--customers", str(DATA_DIR / "customers.csv"),
    "--output-dir", str(OUTPUT_DIR),
    "--max-snapshots", "2",
    "--memory-limit", "6GB",
    "--threads", "2",
    "--metric-sample-size", "2000",
]
print("Running quick test:\n", " ".join(quick_cmd))
subprocess.run(quick_cmd, check=True)


In [ ]:
from pathlib import Path
import pandas as pd, json

print("Output files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        print(f"{p.name:38s} {p.stat().st_size / (1024**2):8.2f} MiB")

metrics_path = OUTPUT_DIR / "clustering_metrics.csv"
if metrics_path.exists():
    display(pd.read_csv(metrics_path))
meta_path = OUTPUT_DIR / "run_metadata.json"
if meta_path.exists():
    print(json.dumps(json.loads(meta_path.read_text()), indent=2)[:5000])


## Full run — chạy sau khi quick test thành công

Cell dưới dùng lại cùng thư mục output nên có thể tái sử dụng `work/customer_day.parquet`; không cần quét lại CSV nếu Parquet trung gian còn nguyên. Nó xử lý tất cả snapshot theo cấu hình mặc định (xấp xỉ 4 tuần/lần). Đây có thể là tác vụ dài, vì vậy nên chạy full run khi bạn đã xác minh quick test và có đủ thời gian runtime.


In [ ]:
# Chạy full experiment. Không chạy cell này nếu quick test còn lỗi.
full_cmd = [
    sys.executable, str(SCRIPT_PATH),
    "--transactions", str(DATA_DIR / "transactions_train.csv"),
    "--customers", str(DATA_DIR / "customers.csv"),
    "--output-dir", str(OUTPUT_DIR),
    "--warmup-days", "180",
    "--snapshot-every-days", "28",
    "--gap-lookback-days", "365",
    "--min-window-days", "28",
    "--max-window-days", "91",
    "--default-window-days", "56",
    "--gap-multiplier", "2.0",
    "--clusters", "5",
    "--metric-sample-size", "2000",
    "--memory-limit", "6GB",
    "--threads", "2",
]
print("Running full experiment:\n", " ".join(full_cmd))
subprocess.run(full_cmd, check=True)


In [ ]:
# Review full-run metrics after the full experiment finishes.
metrics_path = OUTPUT_DIR / "clustering_metrics.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics.sort_values(["snapshot_date", "model"]))
    display(metrics.groupby("model")[["silhouette_sample", "davies_bouldin_sample"]].mean(numeric_only=True))

transitions_path = OUTPUT_DIR / "cluster_transitions.csv"
if transitions_path.exists():
    display(pd.read_csv(transitions_path).head(20))


## Lưu kết quả sang Google Drive (tùy chọn)

Các file trong `/content` sẽ mất khi Colab runtime bị xoá. Cell này chỉ sao chép kết quả nghiên cứu (snapshot Parquet, metric, transition, metadata, model) sang Drive; **không sao chép thư mục `work`** vì nó chứa file DuckDB và Parquet trung gian lớn. Nếu cần chạy tiếp mà không aggregate lại, hãy giữ `work/customer_day.parquet` ở nơi an toàn riêng.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")
DRIVE_OUT = Path("/content/drive/MyDrive/adaptive_rfm_hm_results")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

patterns = ["snapshot_*.parquet", "*.csv", "*.joblib", "*.json"]
copy_list = []
for pattern in patterns:
    copy_list.extend(OUTPUT_DIR.glob(pattern))
for src in sorted(set(copy_list)):
    if src.is_file():
        shutil.copy2(src, DRIVE_OUT / src.name)
        print("Copied:", src.name)
print("Saved result files to:", DRIVE_OUT)


## Cách diễn giải kết quả

- `clustering_metrics.csv`: Silhouette càng cao thường càng tốt; Davies–Bouldin càng thấp thường càng tốt. Đây là metric nội bộ, không phải accuracy và không chứng minh mô hình dự đoán đúng hành vi tương lai.
- `cluster_transitions.csv`: số khách hàng chuyển từ cụm này sang cụm khác giữa hai snapshot liên tiếp, chỉ tính khách hàng xuất hiện ở cả hai snapshot.
- `adaptive_cluster_profile.csv`: thống kê tuổi và các cột hồ sơ có sẵn trong `customers.csv` cho cụm Adaptive ở snapshot cuối.
- `snapshot_*.parquet`: chi tiết RFM và cluster cho từng mốc thời gian.

**Lưu ý nghiên cứu:** so sánh 4/8/13 tuần với adaptive là baseline hợp lý, nhưng bạn vẫn cần đánh giá trên các giai đoạn thời gian giữ lại và tránh tuyên bố phương pháp mới tốt hơn cho đến khi kết quả thực nghiệm hỗ trợ điều đó.
